# Wyklad 2: Programowanie skryptowe w PyQGIS
### Programowanie w GIS — Kurs dla studentów I stopnia

---

> **Kurs:** Programowanie w GIS (QGIS)  
> **Wyklad:** 2 z 7  
> **Tematy:** Python w QGIS | Konsola Pythona | Architektura PyQGIS | Pierwsze skrypty  
> **Wymagania wstepne:** Wyklad 1, podstawowa znajomosc skladni Pythona  

---

### Jak korzystac z tego notatnika

Komorki **Markdown** (takie jak ta) zawieraja teorię i wyjasnienia.  
Komorki **Code** zawieraja przyklady do uruchomienia w Konsoli Pythona QGIS  
lub w edytorze skryptow — **nie uruchamiaj ich bezposrednio w Jupyter**.  

Komorki oznaczone jako `# [JUPYTER]` mozna uruchomic tutaj —  
sprawdzaja srodowisko lub demonstruja skladnie Pythona niezalezna od QGIS.


---
<a id='s1'></a>

## 1. Trzy sposoby programowania w QGIS

Python jest oficjalnym jezykiem skryptowym QGIS. Dostepny jest przez API o nazwie **PyQGIS**,
ktore daje pelny dostep do wszystkich funkcji programu: warstw, geometrii, atrybutow,
algorytmow przetwarzania i interfejsu uzytkownika.

Istnieja trzy glowne konteksty, w ktorych piszemy kod Pythona w ekosystemie QGIS:

### 1.1 Skrypt PyQGIS

Kod uruchamiany wewnatrz dzialajacego QGIS — przez Konsole Pythona lub wbudowany edytor
skryptow. Skrypt ma bezposredni dostep do otwartego projektu, zaladowanych warstw
i algorytmow przetwarzania.

Kiedy uzywac: automatyzacja jednorazowych zadan, przetwarzanie wsadowe plikow, szybkie
operacje na otwartym projekcie.

**Zalety:** natychmiastowy dostep do projektu, prosta skladnia, brak konfiguracji.  
**Ograniczenia:** dziala tylko wewnatrz uruchomionego QGIS, brak wlasnego GUI.

### 1.2 Wtyczka QGIS

Rozszerzenie instalowane przez Menedzera wtyczek, wyposezone we wlasny interfejs graficzny.
Pisana w Pythonie, ladowana przy starcie QGIS, mozna udostepniac przez oficjalne repozytorium.

Kiedy uzywac: narzedzia wielokrotnego uzytku, dystrybucja funkcjonalnosci w zespole lub publicznie.

**Zalety:** wlasny GUI, trwalosc miedzy sesjami, mozliwosc publikacji.  
**Ograniczenia:** wiekszy naklad pracy, wymagana znajomosc Qt i struktury wtyczki.

### 1.3 Biblioteki zewnetrzne (poza QGIS)

Python uruchamiany calkowicie niezaleznie od QGIS — w Jupyterze, terminalu lub IDE.
Zamiast PyQGIS uzywamy bibliotek takich jak `geopandas`, `shapely`, `rasterio`, `pyproj`.

Kiedy uzywac: eksploracja danych, raporty analityczne, integracja GIS z data science.

**Zalety:** niezaleznosc od QGIS, pelna integracja z ekosystemem naukowym Pythona.  
**Ograniczenia:** brak dostepu do projektu QGIS i jego algorytmow.

---

Na tym wykladzie skupiamy sie na **skryptach PyQGIS**. Wtyczki omowimy na Wykladzie 7,
biblioteki zewnetrzne na Wykladach 4 i 5.


In [1]:
# [JUPYTER] Porownianie trzech podejsc — tylko ilustracja skladni

podejscia = [
    {
        'nazwa':        'Skrypt PyQGIS',
        'srodowisko':   'Konsola / edytor w QGIS',
        'kiedy':        'Automatyzacja, przetwarzanie wsadowe',
        'gui':          'Nie',
    },
    {
        'nazwa':        'Wtyczka QGIS',
        'srodowisko':   'Wewnatrz QGIS (ladowana przy starcie)',
        'kiedy':        'Narzedzia wielokrotnego uzytku',
        'gui':          'Tak (Qt)',
    },
    {
        'nazwa':        'Biblioteki zewnetrzne',
        'srodowisko':   'Jupyter / terminal / IDE',
        'kiedy':        'Analiza danych, data science',
        'gui':          'Opcjonalnie',
    },
]

print(f"{'Podejscie':<25} {'Srodowisko':<35} {'GUI':<10}")
print('-' * 72)
for p in podejscia:
    print(f"{p['nazwa']:<25} {p['srodowisko']:<35} {p['gui']:<10}")


Podejscie                 Srodowisko                          GUI       
------------------------------------------------------------------------
Skrypt PyQGIS             Konsola / edytor w QGIS             Nie       
Wtyczka QGIS              Wewnatrz QGIS (ladowana przy starcie) Tak (Qt)  
Biblioteki zewnetrzne     Jupyter / terminal / IDE            Opcjonalnie


---
<a id='s2'></a>

## 2. Konsola Pythona w QGIS

### 2.1 Uruchomienie

Konsole Pythona otwieramy przez:

- menu: `Wtyczki -> Konsola Pythona`
- skrot klawiszowy: `Ctrl+Alt+P`

Konsola sklada sie z dwoch obszarow:

- **obszar wyjscia** (gorny) — wyswietla wyniki i komunikaty bledow
- **wiersz polecen** (dolny) — miejsce wpisywania kodu

### 2.2 Edytor skryptow

Klikniecie ikony *Pokaz edytor* otwiera wbudowany edytor obok konsoli.
Pozwala pisac dluzsze skrypty, zapisywac je jako pliki `.py` i uruchamiac
przyciskiem *Uruchom skrypt*.

| Konsola | Edytor skryptow |
|---|---|
| Wykonanie linii po linii | Wykonanie calego pliku naraz |
| Interaktywna praca | Pisanie kompletnych skryptow |
| Brak zapisu | Zapis do pliku `.py` |
| Szybkie testy | Powtarzalne rozwiazania |

<img src="img\1.png"/>

### 2.3 Obiekty globalne w konsoli QGIS

Wewnatrz Konsoli Pythona dostepne sa dwa specjalne obiekty globalne,
ktorych nie trzeba importowac:

```python
iface       # QgisInterface — dostep do interfejsu uzytkownika QGIS
QgsProject  # klasa projektu — dostep do warstw i ustawien
```

Obiekt `iface` daje dostep do:

```python
iface.activeLayer()   # aktywna warstwa (zaznaczona w Panelu warstw)
iface.mapCanvas()     # obszar mapy
iface.mainWindow()    # glowne okno aplikacji
iface.messageBar()    # pasek komunikatow
```


In [2]:
# [QGIS] Wklej do Konsoli Pythona w QGIS i uruchom
# Sprawdza aktywna warstwe i wyswietla podstawowe informacje

layer = iface.activeLayer()

if layer is None:
    print('Brak aktywnej warstwy. Zaznacz warstwe w Panelu warstw.')
else:
    print(f'Nazwa warstwy : {layer.name()}')
    print(f'Typ obiektu  : {type(layer)}')
    print(f'CRS          : {layer.crs().authid()}')


NameError: name 'iface' is not defined

---
<a id='s3'></a>

## 3. Architektura PyQGIS — moduly i klasy

### 3.1 Glowne moduly

PyQGIS jest podzielony na kilka modulow, z ktorych kazdy odpowiada za inny obszar:

| Modul | Zawartosc | Przykladowe klasy |
|---|---|---|
| `qgis.core` | Logika GIS — warstwy, geometrie, dane, CRS | `QgsProject`, `QgsVectorLayer`, `QgsFeature` |
| `qgis.gui` | Interfejs uzytkownika — widgety, obszar mapy | `QgisInterface`, `QgsMapCanvas` |
| `qgis.analysis` | Algorytmy analityczne | `QgsZonalStatistics`, `QgsInterpolator` |
| `qgis.processing` | Dostep do Processing Toolbox | `processing.run()` |

W skryptach najczesciej pracujemy z modulem `qgis.core`.
W Konsoli Pythona jest on importowany automatycznie.

### 3.2 Konwencja nazewnicza

Wszystkie klasy PyQGIS zaczynaja sie od przedrostka `Qgs`:

```
QgsProject         — projekt QGIS
QgsVectorLayer     — warstwa wektorowa
QgsRasterLayer     — warstwa rastrowa
QgsFeature         — pojedynczy obiekt wektorowy
QgsGeometry        — geometria obiektu (punkt, linia, polygon)
QgsField           — pojedyncze pole atrybutow
QgsFields          — zbior pol atrybutow
QgsPointXY         — punkt 2D (wspolrzedne X, Y)
QgsCoordinateReferenceSystem  — uklad wspolrzednych
```

### 3.3 Hierarchia obiektow

Typowa sciezka dostepu do danych w PyQGIS:

```
QgsProject.instance()
    |
    +-- .mapLayers()         -> slownik {id: QgsMapLayer}
    +-- .mapLayersByName()   -> lista warstw o podanej nazwie
    |
    QgsVectorLayer
        |
        +-- .getFeatures()   -> iterator po obiektach
        |
        QgsFeature
            |
            +-- .geometry()      -> QgsGeometry
            +-- .attributes()    -> lista wartosci
            +-- ['nazwa_pola']   -> wartosc konkretnego pola
```

### 3.4 Typy danych — Python vs C++

Dokumentacja PyQGIS jest generowana z kodu C++. Tabela konwersji typow:

| Typ C++ | Typ Python | Przyklad |
|---|---|---|
| `QString` | `str` | nazwa warstwy |
| `int` | `int` | liczba obiektow |
| `double` | `float` | dlugosc, pole powierzchni |
| `bool` | `bool` | True / False |
| `QList<T>` | `list` | lista warstw |
| `QMap<K,V>` | `dict` | slownik atrybutow |
| `QVariant` | dowolny typ | wartosc atrybutu |


---
<a id='s4'></a>

## 4. Obiekt QgsProject — punkt wejscia do projektu

### 4.1 Singleton

`QgsProject` jest klasa typu **singleton** — istnieje tylko jedna instancja dla calego projektu.
Dostep uzyskujemy zawsze przez metode `.instance()`:

```python
projekt = QgsProject.instance()
```

Nie tworzymy obiektu przez `QgsProject()` — zwrocilaby nowa, pusta instancje
niepowiazana z otwartym projektem.

### 4.2 Informacje o projekcie

```python
projekt = QgsProject.instance()

print(projekt.fileName())       # sciezka do pliku .qgz
print(projekt.homePath())       # folder projektu
print(projekt.title())          # tytul projektu

crs = projekt.crs()
print(crs.authid())             # np. EPSG:4326
print(crs.description())        # pelna nazwa ukladu
```

### 4.3 Pobieranie warstw

```python
projekt = QgsProject.instance()

# Slownik wszystkich warstw {id_warstwy: obiekt_warstwy}
wszystkie = projekt.mapLayers()
print(f'Liczba warstw: {len(wszystkie)}')

# Iteracja po wszystkich warstwach
for layer_id, layer in wszystkie.items():
    print(f'  {layer.name()} — {layer.type()}')

# Pobranie warstwy po nazwie (zwraca liste)
warstwy = projekt.mapLayersByName('kraje')
if warstwy:
    warstwa = warstwy[0]
    print(warstwa.name())
```

**Uwaga:** `mapLayersByName()` zwraca **liste**, bo kilka warstw moze miec ta sama nazwe.
Zawsze sprawdzaj czy lista nie jest pusta przed uzyciem jej pierwszego elementu.

### 4.4 Typy warstw

```python
from qgis.core import QgsMapLayer

for layer in QgsProject.instance().mapLayers().values():
    if layer.type() == QgsMapLayer.VectorLayer:
        print(f'Wektorowa: {layer.name()}')
    elif layer.type() == QgsMapLayer.RasterLayer:
        print(f'Rastrowa:  {layer.name()}')
```


In [3]:
# [JUPYTER] Wzorzec Singleton w czystym Pythonie
# Ilustruje dlaczego QgsProject.instance() zawsze zwraca ten sam obiekt

class Singleton:
    _instancja = None

    def __init__(self, nazwa):
        self.nazwa = nazwa
        self.warstwy = {}

    @classmethod
    def instance(cls):
        if cls._instancja is None:
            cls._instancja = cls('Projekt domyslny')
        return cls._instancja

# Zawsze ten sam obiekt
p1 = Singleton.instance()
p2 = Singleton.instance()

print(f'p1 is p2: {p1 is p2}')   # True — to ten sam obiekt
print(f'id(p1): {id(p1)}')
print(f'id(p2): {id(p2)}')
print()
print('W PyQGIS dziala to identycznie:')
print('  QgsProject.instance() is QgsProject.instance()  ->  True')


p1 is p2: True
id(p1): 1830812886592
id(p2): 1830812886592

W PyQGIS dziala to identycznie:
  QgsProject.instance() is QgsProject.instance()  ->  True


In [ ]:
# [QGIS] Wklej do Konsoli Pythona w QGIS
# Wyswietla informacje o otwartym projekcie i jego warstwach

from qgis.core import QgsProject, QgsMapLayer

projekt = QgsProject.instance()

print('=== Informacje o projekcie ===')
print(f'Tytul     : {projekt.title() or "(brak)"}')
print(f'Plik      : {projekt.fileName() or "(niezapisany)"}')
print(f'CRS       : {projekt.crs().authid()}')
print(f'Warstwy   : {len(projekt.mapLayers())}')
print()

print('=== Lista warstw ===')
for layer in projekt.mapLayers().values():
    typ = 'wektorowa' if layer.type() == QgsMapLayer.VectorLayer else 'rastrowa'
    print(f'  [{typ}] {layer.name()}  ({layer.crs().authid()})')


---
<a id='s5'></a>

## 5. Praca z warstwami wektorowymi

### 5.1 Klasa QgsVectorLayer

`QgsVectorLayer` dziedziczy po `QgsMapLayer` i rozszerza ja o metody
typowe dla danych wektorowych.

```python
layer = iface.activeLayer()

print(layer.name())              # nazwa warstwy
print(layer.source())            # sciezka do pliku
print(layer.crs().authid())      # kod EPSG, np. EPSG:4326
print(layer.featureCount())      # liczba obiektow
print(layer.wkbType())           # typ geometrii

zasieg = layer.extent()
print(f'xMin: {zasieg.xMinimum()}')
print(f'xMax: {zasieg.xMaximum()}')
```

### 5.2 Typy geometrii

| Stala | Opis |
|---|---|
| `QgsWkbTypes.Point` | Punkt |
| `QgsWkbTypes.LineString` | Linia |
| `QgsWkbTypes.Polygon` | Wielokat |
| `QgsWkbTypes.MultiPoint` | Multipunkt |
| `QgsWkbTypes.MultiLineString` | Multilinia |
| `QgsWkbTypes.MultiPolygon` | Multiwielokat |

```python
from qgis.core import QgsWkbTypes

layer = iface.activeLayer()
print(QgsWkbTypes.displayString(layer.wkbType()))  # np. 'Polygon'
```

<img src="img\3.png" />

### 5.3 Pola atrybutow

```python
layer = iface.activeLayer()
pola = layer.fields()

print(f'Liczba pol: {pola.count()}')

for pole in pola:
    print(f'  {pole.name():<20} typ: {pole.typeName()}')
```

Typy pol atrybutow:

| Typ Qt | Opis | Przyklad |
|---|---|---|
| `QString` | Tekst | 'Polska' |
| `int` | Liczba calkowita | 42 |
| `double` | Liczba zmiennoprzecinkowa | 3.14 |
| `QDate` | Data | 2024-01-15 |
| `bool` | Wartosc logiczna | True |


In [ ]:
# [QGIS] Wklej do Konsoli Pythona w QGIS
# Wyswietla szczegolowe informacje o polach aktywnej warstwy

from qgis.core import QgsWkbTypes

layer = iface.activeLayer()

if layer is None:
    print('Brak aktywnej warstwy.')
else:
    print(f'Warstwa    : {layer.name()}')
    print(f'Geometria  : {QgsWkbTypes.displayString(layer.wkbType())}')
    print(f'Obiekty    : {layer.featureCount()}')
    print(f'CRS        : {layer.crs().authid()}')
    print()
    print(f'Pola atrybutow ({layer.fields().count()}):');
    print(f'  {"Nazwa":<25} {"Typ":<15} {"Dlugosc"}')
    print('  ' + '-' * 50)
    for pole in layer.fields():
        print(f'  {pole.name():<25} {pole.typeName():<15} {pole.length()}')


---
<a id='s6'></a>

## 6. Praca z obiektami i atrybutami

### 6.1 Iterator getFeatures()

Metoda `.getFeatures()` zwraca **iterator** po wszystkich obiektach warstwy.
Kazdy obiekt jest instancja klasy `QgsFeature`.

```python
layer = iface.activeLayer()

for obiekt in layer.getFeatures():
    print(obiekt.id())           # numeryczne id obiektu
    print(obiekt.attributes())   # lista wartosci atrybutow
    print(obiekt['name'])        # wartosc konkretnego pola
```

Iterator jest wydajny pamieciowo — obiekty wczytywane sa jeden po jednym,
nie wszystkie naraz. Dla warstw z milionami obiektow jest to kluczowe.

### 6.2 Filtrowanie przez QgsFeatureRequest

Zamiast pobierac wszystkie obiekty i filtrowac je w Pythonie, lepiej przekazac
warunek bezposrednio do bazy danych przez `QgsFeatureRequest`:

```python
from qgis.core import QgsFeatureRequest

layer = iface.activeLayer()

# Filtrowanie przez wyrazenie
req = QgsFeatureRequest().setFilterExpression('"continent" = \'Europe\'')
for obiekt in layer.getFeatures(req):
    print(obiekt['name'])
```

Inne mozliwosci `QgsFeatureRequest`:

```python
from qgis.core import QgsFeatureRequest, QgsRectangle

# Filtrowanie przez zasieg przestrzenny
bbox = QgsRectangle(10.0, 47.0, 25.0, 55.0)
req = QgsFeatureRequest().setFilterRect(bbox)

# Ograniczenie zwracanych pol (szybciej, mniej pamieci)
req = QgsFeatureRequest().setSubsetOfAttributes(['name', 'pop_est'], layer.fields())

# Ograniczenie liczby obiektow
req = QgsFeatureRequest().setLimit(10)

# Laczenie warunkow
req = (QgsFeatureRequest()
       .setFilterExpression('"pop_est" > 1000000')
       .setLimit(5))
```

### 6.3 Dostep do geometrii

```python
layer = iface.activeLayer()

for obiekt in layer.getFeatures():
    geom = obiekt.geometry()

    if geom.type() == 2:   # polygon
        print(f'Pole powierzchni: {geom.area():.2f}')
        print(f'Obwod           : {geom.length():.2f}')

    if geom.type() == 1:   # linia
        print(f'Dlugosc: {geom.length():.2f}')
```

**Uwaga:** `.area()` i `.length()` zwracaja wartosci w jednostkach CRS warstwy.
Dla ukladow geograficznych (stopnie) wyniki nie maja fizycznego sensu —
nalezy wczesniej transformowac geometrie do ukladu metrycznego.

### 6.4 Selekcja programowa

```python
layer = iface.activeLayer()

# Zaznacz obiekty spelniajace warunek
layer.selectByExpression('"continent" = \'Europe\'')
print(f'Zaznaczono: {layer.selectedFeatureCount()} obiektow')

# Pobierz zaznaczone obiekty
for obiekt in layer.selectedFeatures():
    print(obiekt['name'])

# Odznacz wszystko
layer.removeSelection()
```


In [ ]:
# [QGIS] Wklej do Konsoli Pythona w QGIS
# Iteracja po obiektach aktywnej warstwy z filtrowaniem

from qgis.core import QgsFeatureRequest

layer = iface.activeLayer()

if layer is None:
    print('Brak aktywnej warstwy.')
else:
    # Wyswietl pierwsze 5 obiektow
    print(f'Pierwsze 5 obiektow z warstwy: {layer.name()}')
    print('-' * 40)

    req = QgsFeatureRequest().setLimit(5)
    for obiekt in layer.getFeatures(req):
        atrybuty = obiekt.attributes()
        print(f'  id={obiekt.id():>5} | {atrybuty}')

    print()
    print(f'Lacznie obiektow w warstwie: {layer.featureCount()}')


---
<a id='s7'></a>

## 7. Zapis i modyfikacja danych

### 7.1 Tryb edycji

Modyfikacja atrybutow i geometrii wymaga przelaczenia warstwy w **tryb edycji**.
Jest to ta sama operacja co klikniecie przycisku *Przelacz edycje* w interfejsie QGIS.

```python
layer = iface.activeLayer()

layer.startEditing()     # wlacz tryb edycji

# ... wprowadz zmiany ...

layer.commitChanges()    # zatwierdz i zapisz do pliku
# LUB
layer.rollBack()         # odrzuc zmiany
```

Schemat pracy z edycja:

```
startEditing()
    |
    +-- changeAttributeValue()  — zmiana wartosci atrybutu
    +-- changeGeometry()        — zmiana geometrii
    +-- addFeature()            — dodanie nowego obiektu
    +-- deleteFeatures()        — usuniecie obiektow
    |
    +-- commitChanges()         -> zapis trwaly
    lub rollBack()               -> cofniecie zmian
```

### 7.2 Zmiana wartosci atrybutu

```python
layer = iface.activeLayer()
layer.startEditing()

for obiekt in layer.getFeatures():
    fid = obiekt.id()
    idx = layer.fields().indexOf('nazwa_pola')
    nowa_wartosc = obiekt['stare_pole'].upper()
    layer.changeAttributeValue(fid, idx, nowa_wartosc)

layer.commitChanges()
print('Zmiany zapisane.')
```

### 7.3 Dodawanie nowego obiektu

```python
from qgis.core import QgsFeature, QgsGeometry, QgsPointXY

layer = iface.activeLayer()
layer.startEditing()

nowy = QgsFeature(layer.fields())
nowy.setGeometry(QgsGeometry.fromPointXY(QgsPointXY(21.01, 52.23)))
nowy['name'] = 'Nowy punkt'
nowy['pop_est'] = 1800000

layer.addFeature(nowy)
layer.commitChanges()
```

### 7.4 Usuwanie obiektow

```python
layer = iface.activeLayer()
layer.startEditing()

do_usuniecia = [ob.id() for ob in layer.getFeatures()
                if ob['pop_est'] < 1000]

layer.deleteFeatures(do_usuniecia)
layer.commitChanges()
print(f'Usunieto {len(do_usuniecia)} obiektow.')
```

### 7.5 Odswiezanie widoku

```python
layer.triggerRepaint()
iface.mapCanvas().refresh()
```


In [5]:
# [JUPYTER] Ilustracja wzorca trybu edycji w czystym Pythonie
# Pokazuje analogiczny mechanizm commit/rollback

class WarstwaDemo:
    def __init__(self, obiekty):
        self._dane = list(obiekty)       # dane zatwierdzone
        self._bufor = None               # bufor edycji
        self._tryb_edycji = False

    def startEditing(self):
        self._bufor = [dict(o) for o in self._dane]
        self._tryb_edycji = True
        print('Tryb edycji: WLACZONY')

    def commitChanges(self):
        self._dane = self._bufor
        self._bufor = None
        self._tryb_edycji = False
        print('Zmiany ZATWIERDZONE.')

    def rollBack(self):
        self._bufor = None
        self._tryb_edycji = False
        print('Zmiany ODRZUCONE.')

    def changeAttr(self, idx, pole, wartosc):
        if not self._tryb_edycji:
            raise RuntimeError('Najpierw wywolaj startEditing()')
        self._bufor[idx][pole] = wartosc

    def getFeatures(self):
        dane = self._bufor if self._tryb_edycji else self._dane
        return iter(dane)


# Demonstracja
warstwa = WarstwaDemo([
    {'id': 1, 'name': 'polska',  'pop': 38_000_000},
    {'id': 2, 'name': 'niemcy',  'pop': 84_000_000},
])

print('Przed edycja:')
for ob in warstwa.getFeatures():
    print(f'  {ob}')

warstwa.startEditing()
warstwa.changeAttr(0, 'name', 'POLSKA')   # zmiana na wielkie litery
warstwa.changeAttr(1, 'name', 'NIEMCY')

print('W trakcie edycji (bufor):')
for ob in warstwa.getFeatures():
    print(f'  {ob}')

warstwa.commitChanges()

print('Po zatwierdzeniu:')
for ob in warstwa.getFeatures():
    print(f'  {ob}')


Przed edycja:
  {'id': 1, 'name': 'polska', 'pop': 38000000}
  {'id': 2, 'name': 'niemcy', 'pop': 84000000}
Tryb edycji: WLACZONY
W trakcie edycji (bufor):
  {'id': 1, 'name': 'POLSKA', 'pop': 38000000}
  {'id': 2, 'name': 'NIEMCY', 'pop': 84000000}
Zmiany ZATWIERDZONE.
Po zatwierdzeniu:
  {'id': 1, 'name': 'POLSKA', 'pop': 38000000}
  {'id': 2, 'name': 'NIEMCY', 'pop': 84000000}


---
<a id='s8'></a>

## 8. Edytor skryptow — pierwsze kompletne skrypty

### 8.1 Struktura dobrego skryptu PyQGIS

Dobry skrypt powinien:

1. sprawdzac czy dane wejsciowe sa dostepne i poprawne,
2. informowac uzytkownika o postepie przez komunikaty,
3. obslugiwac bledy przez blok `try/except`,
4. na koncu odswiezac widok mapy.

### 8.2 Skrypt 1: Raport o warstwach projektu


In [ ]:
# [QGIS] Skrypt 1 — raport_warstw.py
# Generuje raport o wszystkich warstwach otwartego projektu.
# Wklej do edytora skryptow QGIS i uruchom.

from qgis.core import QgsProject, QgsMapLayer, QgsWkbTypes

def raport_warstw():
    projekt = QgsProject.instance()
    warstwy = list(projekt.mapLayers().values())

    print('=' * 55)
    print(f'  Projekt : {projekt.title() or "(bez nazwy)"}')
    print(f'  Plik    : {projekt.fileName() or "(niezapisany)"}')
    print(f'  CRS     : {projekt.crs().authid()}')
    print(f'  Warstwy : {len(warstwy)}')
    print('=' * 55)

    for layer in warstwy:
        print(f'\n  Warstwa: {layer.name()}')

        if layer.type() == QgsMapLayer.VectorLayer:
            geom_typ = QgsWkbTypes.displayString(layer.wkbType())
            print(f'    Typ geometrii : {geom_typ}')
            print(f'    Liczba obiektow: {layer.featureCount()}')
            print(f'    Liczba pol    : {layer.fields().count()}')
            nazwy = [p.name() for p in layer.fields()]
            print(f'    Pola          : {', '.join(nazwy)}')
        elif layer.type() == QgsMapLayer.RasterLayer:
            print(f'    Pasma           : {layer.bandCount()}')
            print(f'    Rozdzielczosc X : {layer.rasterUnitsPerPixelX():.4f}')

        print(f'    CRS    : {layer.crs().authid()}')
        print(f'    Zrodlo : {layer.source()}')

    print('\n' + '=' * 55)
    print('  Raport zakonczony.')
    print('=' * 55)

raport_warstw()


### 8.3 Skrypt 2: Eksport atrybutow do pliku CSV


In [ ]:
# [QGIS] Skrypt 2 — eksport_csv.py
# Eksportuje atrybuty aktywnej warstwy do pliku CSV.
# Zmien sciezke_wyjsciowa przed uruchomieniem.

import csv
from qgis.core import QgsMapLayer, QgsFeatureRequest

def eksportuj_do_csv(sciezka_wyjsciowa):
    layer = iface.activeLayer()

    if layer is None:
        print('Blad: brak aktywnej warstwy.')
        return

    if layer.type() != QgsMapLayer.VectorLayer:
        print('Blad: aktywna warstwa nie jest wektorowa.')
        return

    nazwy_pol = [pole.name() for pole in layer.fields()]
    n = layer.featureCount()
    print(f'Eksport: {layer.name()}  ({n} obiektow, {len(nazwy_pol)} pol)')

    try:
        with open(sciezka_wyjsciowa, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(nazwy_pol)

            for i, obiekt in enumerate(layer.getFeatures()):
                writer.writerow(obiekt.attributes())
                if (i + 1) % 500 == 0:
                    print(f'  Przetworzono: {i + 1}/{n}')

        print(f'Gotowe. Plik zapisany: {sciezka_wyjsciowa}')

    except IOError as e:
        print(f'Blad zapisu pliku: {e}')


# Zmien sciezke na swoja
eksportuj_do_csv('C:/GIS_kurs/eksport.csv')


In [ ]:
# [JUPYTER] Wersja demonstracyjna eksportu CSV — dziala w Jupyter
# Uzywa danych testowych zamiast warstwy QGIS

import csv
import io

# Symulowane dane warstwy
pola = ['id', 'name', 'continent', 'pop_est']
obiekty = [
    [1, 'Polska',   'Europe', 38_000_000],
    [2, 'Niemcy',   'Europe', 84_000_000],
    [3, 'Francja',  'Europe', 68_000_000],
    [4, 'Nigeria',  'Africa', 220_000_000],
    [5, 'Brazylia', 'South America', 215_000_000],
]

# Zapis do strumienia tekstowego (zamiast pliku)
bufor = io.StringIO()
writer = csv.writer(bufor)
writer.writerow(pola)
for obiekt in obiekty:
    writer.writerow(obiekt)

print('Zawartosc wygenerowanego CSV:')
print('-' * 45)
print(bufor.getvalue())

# W QGIS zastap bufor = io.StringIO() przez:
# with open('sciezka.csv', 'w', newline='', encoding='utf-8') as bufor:


---
<a id='s9'></a>

## 9. Podsumowanie i slownik pojec

### 9.1 Co omowilismy na wykladzie 2

- trzy konteksty programowania w Pythonie w ekosystemie QGIS: skrypt, wtyczka, biblioteki zewnetrzne,
- Konsole Pythona i edytor skryptow jako srodowisko pracy,
- glowne moduly PyQGIS: `qgis.core`, `qgis.gui`, `qgis.analysis`, `qgis.processing`,
- singleton `QgsProject.instance()` jako punkt wejscia do projektu,
- pobieranie i filtrowanie warstw przez `mapLayers()` i `mapLayersByName()`,
- iteracje po obiektach warstwy przez `getFeatures()` i `QgsFeatureRequest`,
- dostep do geometrii i atrybutow obiektow klasy `QgsFeature`,
- programowa selekcje obiektow,
- modyfikacje danych w trybie edycji: zmiana atrybutow, dodawanie i usuwanie obiektow,
- strukture kompletnego skryptu PyQGIS.

### 9.2 Slownik kluczowych pojec

| Pojecie | Definicja |
|---|---|
| **PyQGIS** | API Pythona dla QGIS — zestaw klas i modulow umozliwiajacych programowanie wszystkich funkcji QGIS |
| **API** | Application Programming Interface — interfejs programistyczny, zbior klas i metod udostepnionych przez aplikacje |
| **Singleton** | Wzorzec projektowy, w ktorym klasa ma tylko jedna instancje; `QgsProject.instance()` zawsze zwraca ten sam obiekt |
| **`QgsProject`** | Klasa reprezentujaca otwarty projekt QGIS; zawiera wszystkie warstwy i ustawienia |
| **`QgsVectorLayer`** | Klasa reprezentujaca warstwe wektorowa; daje dostep do obiektow, atrybutow i geometrii |
| **`QgsFeature`** | Pojedynczy obiekt warstwy wektorowej (wiersz w tabeli atrybutow + geometria) |
| **`QgsGeometry`** | Geometria obiektu; daje dostep do metod pomiarowych i topologicznych |
| **`QgsFeatureRequest`** | Klasa do definiowania filtrow przy pobieraniu obiektow |
| **`iface`** | Globalny obiekt `QgisInterface` w konsoli QGIS; daje dostep do interfejsu uzytkownika |
| **tryb edycji** | Stan warstwy umozliwiajacy modyfikacje danych; startEditing() / commitChanges() |
| **iterator** | Obiekt zwracajacy elementy jeden po jednym; getFeatures() jest iteratorem, nie lista |

### 9.3 Na Wykladzie 3

- operacje geometryczne: buforowanie, przeciecia, roznica, scalanie,
- reprojekcja geometrii miedzy ukladami wspolrzednych,
- tworzenie nowych warstw wektorowych od podstaw,
- zapis warstw do pliku (GeoPackage, Shapefile, GeoJSON),
- uruchamianie algorytmow Processing Toolbox z poziomu skryptu.

---

## Literatura i zasoby

- PyQGIS Developer Cookbook: https://docs.qgis.org/latest/en/docs/pyqgis_developer_cookbook/
- API Reference: https://api.qgis.org/api/
- QgsVectorLayer: https://api.qgis.org/api/classQgsVectorLayer.html
- Spatial Thoughts — kurs PyQGIS: https://courses.spatialthoughts.com/pyqgis-in-a-day.html

---
*Wyklad 2 — Programowanie w GIS (QGIS) — studia licencjackie.*


In [11]:
# [JUPYTER] Krotki quiz podsumowujacy — sprawdz wiedze z wykladu

pytania = [
    {
        'pytanie': 'Jak uzyskac dostep do otwartego projektu QGIS?',
        'odpowiedz': 'QgsProject.instance()',
    },
    {
        'pytanie': 'Jaka metoda zwraca iterator po obiektach warstwy?',
        'odpowiedz': 'layer.getFeatures()',
    },
    {
        'pytanie': 'Jak wlaczyc tryb edycji warstwy?',
        'odpowiedz': 'layer.startEditing()',
    },
    {
        'pytanie': 'Co zwraca mapLayersByName() — liste czy slownik?',
        'odpowiedz': 'Liste (list)',
    },
    {
        'pytanie': 'Jakiego przedrostka uzywaja wszystkie klasy PyQGIS?',
        'odpowiedz': 'Qgs (np. QgsVectorLayer, QgsFeature)',
    },
]

for n in range(10):
    print('\n')
print('=== Quiz: Wyklad 2 — PyQGIS ===')
for i, q in enumerate(pytania, 1):
    print(f'  {i}. {q["pytanie"]}')






















=== Quiz: Wyklad 2 — PyQGIS ===
  1. Jak uzyskac dostep do otwartego projektu QGIS?
  2. Jaka metoda zwraca iterator po obiektach warstwy?
  3. Jak wlaczyc tryb edycji warstwy?
  4. Co zwraca mapLayersByName() — liste czy slownik?
  5. Jakiego przedrostka uzywaja wszystkie klasy PyQGIS?
